## Het feit rit S2

In [1]:
#SETUP
import ConnectionConfig as cc
cc.setupEnvironment()
from pyspark.sql import SparkSession

In [2]:
#SETUP
spark = cc.startLocalCluster("factRidesSaif")
spark.getActiveSession()


In [3]:
#EXTRACT
cc.set_connectionProfile("VeloDB")

# ride_source_df = spark.read \
#     .format("jdbc") \
#     .option("url", cc.create_jdbc()) \
#     .option("driver", cc.get_Property("driver")) \
#     .option("dbtable", "rides") \
#     .option("user", cc.get_Property("username")) \
#     .option("password", cc.get_Property("password")) \
#     .option("partitionColumn", "rideId") \
#     .option("numPartitions", 4) \
#     .option("lowerBound", 0) \
#     .option("upperBound", 1000) \
#     .load()

ride_source_df = spark.read \
    .format("jdbc") \
    .option("url", cc.create_jdbc()) \
    .option("driver", cc.get_Property("driver")) \
    .option("query", """
        SELECT *,
            haversine_km(
                CAST(r.startpoint[0] AS NUMERIC), 
                CAST(r.startpoint[1] AS NUMERIC),
                CAST(r.endpoint[0] AS NUMERIC), 
                CAST(r.endpoint[1] AS NUMERIC)
            ) AS distance
        FROM rides r
    """) \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

subscription_source_df = spark.read \
    .format("jdbc") \
    .option("url", cc.create_jdbc()) \
    .option("driver", cc.get_Property("driver")) \
    .option("dbtable", "subscriptions") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "subscriptionid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 1000) \
    .load()

customers_source_df = spark.read \
    .format("jdbc") \
    .option("url", cc.create_jdbc()) \
    .option("driver", cc.get_Property("driver")) \
    .option("dbtable", "velo_users") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "userid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 1000) \
    .load()


In [4]:
ride_source_df.createOrReplaceTempView("rides_source_saif")
subscription_source_df.createOrReplaceTempView("subscription_source")
customers_source_df.createOrReplaceTempView("customers_source")

In [5]:

rides_with_users = spark.sql("""
SELECT r.*, c.userid FROM rides_source_saif r
LEFT JOIN subscription_source s ON r.subscriptionid = s.subscriptionid
left join customers_source c on s.userid = c.userid""")


In [6]:
#EXTRACT
dim_customer = spark.read.format("delta").load("spark-warehouse/dimCustomer")
dim_locks = spark.read.format("delta").load("spark-warehouse/dimLocks")

In [7]:
#TRANSFORM
dim_customer.createOrReplaceTempView("dimCustomer")
dim_locks.createOrReplaceTempView("dimLocks")
rides_with_users.createOrReplaceTempView("rides_with_users")


In [8]:
#TRANSFORM
fact_rides = spark.sql("""
    SELECT 
        monotonically_increasing_id() as rideSK, 
        rideid AS rideId,
        l1.lockSK AS startLockSK,
        l2.lockSK AS endLockSK,
        c.customer_SK AS customerSK,
        r.distance  as rideDistance
        
    FROM
        rides_with_users AS r
    LEFT OUTER JOIN 
        dimLocks AS l1 ON r.startlockId = l1.lockId
    LEFT OUTER JOIN 
        dimLocks AS l2 ON r.endlockId = l2.lockId
    LEFT OUTER JOIN 
        dimCustomer AS c ON (r.userid = c.customer_id
        and r.starttime >= c.scd_start and (r.starttime < c.scd_end  OR c.scd_end is null)
        )
    WHERE 
        l1.lockId IS NOT NULL AND l2.lockId IS NOT NULL AND c.customer_id IS NOT NULL
        
""")

In [9]:
#LOAD
fact_rides.show(20)
# fact_rides.write.format("delta").mode("overwrite").saveAsTable("factRidesS2")

+------+-------+-----------+---------+----------+--------------------+
|rideSK| rideId|startLockSK|endLockSK|customerSK|        rideDistance|
+------+-------+-----------+---------+----------+--------------------+
|     0|3408972|       5286|     6317|8589972324|4.038272935078430000|
|     1|3408982|       4891|     5634|8589966248|2.371930324993150000|
|     2|3408996|         49|     1003|8589981203|2.002955676810820000|
|     3|3408997|       4099|     2695|8589944065|3.167369001512150000|
|     4|3408999|       5976|      696|8589956306|2.859825631258500000|
|     5|3408999|       5976|      696|8589956305|2.859825631258500000|
|     6|3409000|       6497|     5976|8589944069|2.060463678496970000|
|     7|3409004|       2137|     1831|8589952262|1.102862834833480000|
|     8|3409007|       1759|       67|8589971140|3.872382206062740000|
|     9|3409015|       6137|     2389|8589978250|1.731712964361570000|
|    10|3409017|       5795|     1434|8589973140|2.488040473160680000|
|    1

In [10]:
delta_table_path = "spark-warehouse/factRidesS2"
fact_rides.write.format("delta").mode("overwrite").save(delta_table_path)

In [11]:
spark.stop()